# Bulgarian phoneme FastSpeech2 — local inference

This notebook runs the punctuation-aware Bulgarian phoneme model locally and lets you switch between the repo's default universal HiFi-GAN vocoder and your fine-tuned HiFi-GAN generator.

Run it from the repository root, or keep `REPO_ROOT` below pointed at the repository.

In [1]:
import os, shutil, subprocess, sys, zipfile
from datetime import datetime
from pathlib import Path

from IPython.display import Audio, display

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / 'synthesize.py').is_file() else cwd.parent
assert (REPO_ROOT / 'synthesize.py').is_file(), f'Could not find repo root from {cwd}'
os.chdir(REPO_ROOT)

from bg_text_normalizer import normalize as contextual_normalize
from bulgarian_normalization import normalize_for_mfa, normalize_with_punctuation, prosody_words
from num2wordBg import text_numbers_to_words

print('repo root:', REPO_ROOT)

repo root: D:\FastSpeech2\FastSpeech2


In [2]:
CHECKPOINT = None
CKPT_DIR = 'output/ckpt/Bulgarian'
RESTORE_STEP = None

FINETUNED_VOCODER = 'hifigan_finetune/g_00080000'

DURATION_CONTROL = 1.0
PITCH_CONTROL = 1.0
ENERGY_CONTROL = 1.0

USE_CUSTOM_TEXT_NORMALIZER = True
TEXT_NORMALIZER_MODE = 'contextual'
SHOW_NORMALIZATION_PREVIEW = True

MFA_CMD = 'mfa'
MFA_BIN = None
MFA_ROOT_DIR = None
MAMBA_ROOT_PREFIX = None

PHONEME_ASSETS_ZIP = 'local_assets/phoneme_assets.zip'
PREPROCESSED_ZIP = 'local_assets/preprocessed_Bulgarian_prosody_v2.zip'

RESULT_DIR = 'local_inference_results'
Path(RESULT_DIR).mkdir(parents=True, exist_ok=True)

## Download the trained files from Google Drive

Before running the asset-check cells below, download the inference artifacts from the Google Drive folder used by the A100 training notebook. The default Drive folder is:

```text
MyDrive/fs2_bg_phone/
```

Put the downloaded files at these exact paths relative to the local `FastSpeech2/` repository root:

| Google Drive file | Local path |
| --- | --- |
| `fs2_bg_phone/phoneme_assets.zip` | `local_assets/phoneme_assets.zip` |
| `fs2_bg_phone/preprocessed_Bulgarian_prosody_v2.zip` | `local_assets/preprocessed_Bulgarian_prosody_v2.zip` |
| `fs2_bg_phone/output_prosody_v2/ckpt/Bulgarian/60000.pth.tar` or another numbered checkpoint | `output/ckpt/Bulgarian/60000.pth.tar` |
| `fs2_bg_phone/hifigan_finetune/g_00135000` (optional, for `run_inference('finetuned', text)`) | `hifigan_finetune/g_00135000` |

Create the folders if they do not exist:

```bash
mkdir -p local_assets output/ckpt/Bulgarian hifigan_finetune
```

Do not manually extract the two zip files. This notebook unpacks `phoneme_assets.zip` into the repo root and extracts only the needed metadata from `preprocessed_Bulgarian_prosody_v2.zip`. If you use a checkpoint name other than `60000.pth.tar`, either keep it inside `output/ckpt/Bulgarian/` and let `CKPT_DIR` find the latest checkpoint, or set `CHECKPOINT` to the exact local file path.

If Drive contains an updated `g2p_cache_prosody_v2.json`, it is optional. After the asset restore cell has run, you can copy it to `preprocessed_data/Bulgarian/g2p_cache.json`; otherwise unknown words can still be handled through MFA G2P when `MFA_CMD` points to a working MFA installation.

In [3]:
INSTALL_PYTHON_DEPS = False

if INSTALL_PYTHON_DEPS:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
    print('Python dependencies installed')
else:
    print('Skipping pip install')

Skipping pip install


In [4]:
phoneme_assets_zip = Path(PHONEME_ASSETS_ZIP)
preprocessed_zip = Path(PREPROCESSED_ZIP)

if phoneme_assets_zip.is_file():
    shutil.unpack_archive(str(phoneme_assets_zip), '.')
    print('unpacked:', phoneme_assets_zip)
else:
    print('not found, skipping:', phoneme_assets_zip)

stats = Path('preprocessed_data/Bulgarian/stats.json')
if preprocessed_zip.is_file() and not stats.is_file():
    wanted_suffixes = {
        'Bulgarian/stats.json',
        'Bulgarian/speakers.json',
        'Bulgarian/linguistic_abi.json',
        'Bulgarian/g2p_cache.json',
    }
    with zipfile.ZipFile(preprocessed_zip) as zf:
        for member in zf.namelist():
            normalized = member.strip('/')
            if any(normalized.endswith(suffix) for suffix in wanted_suffixes):
                if normalized.startswith('preprocessed_data/'):
                    zf.extract(member, '.')
                else:
                    zf.extract(member, 'preprocessed_data')
    print('extracted minimal preprocessed metadata from:', preprocessed_zip)
elif stats.is_file():
    print('stats already present:', stats)
else:
    print('not found, skipping:', preprocessed_zip)

unpacked: local_assets\phoneme_assets.zip
stats already present: preprocessed_data\Bulgarian\stats.json


In [5]:
required = [
    Path('lexicon/bulgarian_mfa_runtime.dict'),
    Path('preprocessed_data/Bulgarian/stats.json'),
    Path('hifigan/generator_universal.pth.tar.zip'),
]
for path in required:
    assert path.is_file(), f'Missing {path}'

if CHECKPOINT:
    assert Path(CHECKPOINT).is_file(), f'Missing checkpoint {CHECKPOINT}'
else:
    ckpts = sorted(Path(CKPT_DIR).glob('*.pth.tar'))
    assert ckpts, f'No checkpoints found in {CKPT_DIR}'
    print('latest checkpoint:', ckpts[-1])

if Path(FINETUNED_VOCODER).is_file():
    print('fine-tuned vocoder found:', FINETUNED_VOCODER)
else:
    print('fine-tuned vocoder not found yet:', FINETUNED_VOCODER)

print('basic local inference assets OK')

latest checkpoint: output\ckpt\Bulgarian\60000.pth.tar
fine-tuned vocoder found: hifigan_finetune/g_00080000
basic local inference assets OK


In [6]:
def normalize_text_for_inference(text):
    if not USE_CUSTOM_TEXT_NORMALIZER or TEXT_NORMALIZER_MODE == 'none':
        return text
    if TEXT_NORMALIZER_MODE == 'legacy':
        return normalize_with_punctuation(text)
    if TEXT_NORMALIZER_MODE == 'contextual':
        return normalize_with_punctuation(contextual_normalize(text))
    raise ValueError("TEXT_NORMALIZER_MODE must be 'contextual', 'legacy', or 'none'")


def preview_text_normalization(text):
    normalized = normalize_text_for_inference(text)
    if not SHOW_NORMALIZATION_PREVIEW:
        return normalized
    print('Input text:')
    print(' ', text)
    print('Contextual normalizer output:')
    print(' ', contextual_normalize(text))
    print('Legacy number-expanded only via num2wordBg:')
    print(' ', text_numbers_to_words(text))
    print('Normalized text that tools/infer_bulgarian.py will synthesize:')
    print(' ', normalized)
    print('MFA word-only text after selected normalization:')
    print(' ', normalize_for_mfa(normalized))
    punctuation_tokens = [token for _, token in prosody_words(normalized) if token]
    print('Punctuation/control tokens after selected normalization:')
    print(' ', punctuation_tokens or '(none)')
    return normalized


def run_inference(vocoder_mode, text, *, output_id=None):
    vocoder_mode = vocoder_mode.lower().strip()
    preview_text_normalization(text)
    output_id = output_id or f"infer_{vocoder_mode}_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}"
    cmd = [
        sys.executable,
        'tools/infer_bulgarian.py',
        '--text', text,
        '--output-id', output_id,
        '--result-dir', RESULT_DIR,
        '--duration-control', str(DURATION_CONTROL),
        '--pitch-control', str(PITCH_CONTROL),
        '--energy-control', str(ENERGY_CONTROL),
        '--vocoder-mode', vocoder_mode,
        '--text-normalizer', TEXT_NORMALIZER_MODE if USE_CUSTOM_TEXT_NORMALIZER else 'none',
        '--mfa-cmd', MFA_CMD,
    ]
    if CHECKPOINT:
        cmd += ['--checkpoint', CHECKPOINT]
    else:
        cmd += ['--ckpt-dir', CKPT_DIR]
    if RESTORE_STEP is not None:
        cmd += ['--restore-step', str(RESTORE_STEP)]
    if vocoder_mode != 'default':
        cmd += ['--finetuned-vocoder', FINETUNED_VOCODER]
    if MFA_BIN:
        cmd += ['--mfa-bin', MFA_BIN]
    if MFA_ROOT_DIR:
        cmd += ['--mfa-root-dir', MFA_ROOT_DIR]
    if MAMBA_ROOT_PREFIX:
        cmd += ['--mamba-root-prefix', MAMBA_ROOT_PREFIX]

    print('RUN:', ' '.join(map(str, cmd)))
    result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    result.check_returncode()

    wav = Path(RESULT_DIR) / f'{output_id}.wav'
    png = Path(RESULT_DIR) / f'{output_id}.png'
    assert wav.is_file(), wav
    print('wav:', wav)
    if png.is_file():
        print('plot:', png)
    display(Audio(str(wav), rate=22050))
    return wav

In [7]:
text = "Чете Здравно Димитров"
wav_default = run_inference('finetuned', text)

Input text:
  Чете Здравно Димитров
Contextual normalizer output:
  Чете Здравно Димитров
Legacy number-expanded only via num2wordBg:
  Чете Здравно Димитров
Normalized text that tools/infer_bulgarian.py will synthesize:
  чете здравно димитров
MFA word-only text after selected normalization:
  чете здравно димитров
Punctuation/control tokens after selected normalization:
  (none)
RUN: c:\Users\stani\AppData\Local\Programs\Python\Python313\python.exe tools/infer_bulgarian.py --text Чете Здравно Димитров --output-id infer_finetuned_20260625_230435_533048 --result-dir local_inference_results --duration-control 1.0 --pitch-control 1.0 --energy-control 1.0 --vocoder-mode finetuned --text-normalizer contextual --mfa-cmd mfa --ckpt-dir output/ckpt/Bulgarian --finetuned-vocoder hifigan_finetune/g_00080000
Traceback (most recent call last):
  File "D:\FastSpeech2\FastSpeech2\tools\infer_bulgarian.py", line 26, in <module>
    import torch
ModuleNotFoundError: No module named 'torch'



CalledProcessError: Command '['c:\\Users\\stani\\AppData\\Local\\Programs\\Python\\Python313\\python.exe', 'tools/infer_bulgarian.py', '--text', 'Чете Здравно Димитров', '--output-id', 'infer_finetuned_20260625_230435_533048', '--result-dir', 'local_inference_results', '--duration-control', '1.0', '--pitch-control', '1.0', '--energy-control', '1.0', '--vocoder-mode', 'finetuned', '--text-normalizer', 'contextual', '--mfa-cmd', 'mfa', '--ckpt-dir', 'output/ckpt/Bulgarian', '--finetuned-vocoder', 'hifigan_finetune/g_00080000']' returned non-zero exit status 1.